# Building Gym Environment for Reinforcement Learning (Pure PnL Version)

This notebook will build a custom OpenAI Gym environment based on the hftbacktest framework and PMM strategy for training reinforcement learning agents to optimize market making strategies.

## Environment Design Goals

-   **State Space**: Order book features, price changes, position status, PnL, etc.
-   **Action Space**: Spread adjustments, order quantities, strategy parameter adjustments, etc.
-   **Reward Function**: Multi-dimensional reward design based on PnL changes, risk control, etc.
-   **Environment**: High-fidelity simulation environment based on real market data
-   **Design Philosophy**: Fully follows HFTBacktest native design, PnL starts from 0

In [ ]:
import torch
import warnings

# TorchRL related imports
from tensordict import TensorDict

# Set warning filters
warnings.filterwarnings('ignore')

# Set default device (priority: CUDA > MPS > CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("🚀 Using CUDA GPU acceleration")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🍎 Using Apple Silicon MPS acceleration")
else:
    device = torch.device("cpu")
    print("💻 Using CPU")

print(f"Environment dependencies imported successfully, using device: {device}")

In [ ]:
# Test PMM Reinforcement Learning Environment

# Import our built PMM environment
from lib.rl_env import create_pmm_env
from hftbacktest import BacktestAsset
import numpy as np
import os

# Use a single test data file
# test_data_file = "data/output/xrpusdt_20250717.npz"
test_data_file = "data/slices/xrpusdt_20250717_2h_77562432/split_003_2h.npz"

if not os.path.exists(test_data_file):
    raise FileNotFoundError(
        f"❌ Test data file does not exist: {test_data_file}\n   Please ensure the data file exists or modify the test_data_file path")

print(f"✅ Found test data file: {test_data_file}")

try:
    print(f"🔄 Loading data: {test_data_file}")

    # Important: Load numpy array first, then pass to BacktestAsset
    test_data = np.load(test_data_file)['data']
    print(f"✅ Data loaded successfully, shape: {test_data.shape}")

    # Create BacktestAsset correctly following 03_strategy_design.ipynb
    data_asset = (
        BacktestAsset()
        .data([test_data])  # Pass numpy array, not file path!
        .linear_asset(1.0)       # Linear asset, contract multiplier of 1
        .risk_adverse_queue_model()  # Risk adverse queue model
        .no_partial_fill_exchange()  # No partial fill allowed
        .constant_latency(10_000_000, 10_000_000)  # Add latency settings
        .tick_size(0.0001)         # XRP minimum price precision (0.0001 USDT)
        .lot_size(0.1)         # XRP minimum trade quantity
        .trading_value_fee_model(-0.00003, 0.0007)  # Binance fee model
        .power_prob_queue_model3(3.0)  # Add queue model
    )

    print(f"✅ Successfully created BacktestAsset")
    print(f"   Data shape: {test_data.shape}")
    print(f"   Configuration completed")
    print(f"   📌 Using pure PnL environment (HFTBacktest native design)")

except Exception as e:
    raise RuntimeError(f"❌ Failed to create BacktestAsset: {e}") from e

## 🎯 PMM Reinforcement Learning Environment Description

### Environment Features

We have successfully built a TorchRL reinforcement learning environment based on pure PMM strategy (pmm_pure.py), using **pure PnL design** that fully follows HFTBacktest's native philosophy.

#### 🎮 **Action Space** (4-dimensional continuous actions)

1. **half_spread** (1-50): Half spread (number of ticks)
2. **skew** (1-50): Skew coefficient (number of ticks), tick offset per normalized position unit
3. **grid_num** (1-10): Number of grid order layers
4. **grid_interval** (1-50): Grid interval (number of ticks)

**Notes**:

-   `skew` is now in tick units, providing standardized behavior across assets
-   `order_qty_dollar` is set to a fixed value of $50.0 as it mainly affects capital management rather than core strategy logic

#### 👁️ **Observation Space** (4-dimensional state vector)

1. **Normalized mid price**: Current market mid price / 1000
2. **Spread**: (ask-bid)/mid_price \* 1000 (amplified signal)
3. **Normalized position**: Current position / 1000
4. **PnL**: Cumulative profit and loss (USD)

#### 🏆 **Reward Function** (Most direct design)

-   **PnL reward**: Direct use of PnL change (earn $1 get reward 1, lose $1 get reward -1)
-   **Risk penalty**: Penalty based on actual position value (penalty starts when position value exceeds $1000)
-   **No trading costs**: In negative fee environment, rebates are already included in PnL

#### 💡 **Design Philosophy**

-   **No initial_balance**: HFTBacktest natively starts PnL recording from 0
-   **Direct PnL usage**: Earn what you get, the most direct reward signal
-   **Value-based risk control**: Use actual position value rather than abstract ratios
-   **KISS principle**: Keep it simple and direct, avoid unnecessary transformations

### Strategy Description

Uses pure PMM market making strategy from `pmm_pure.py`:

-   Calculate base price based on mid price
-   Adjust reservation price according to position risk (skew \* tick_size \* normalized_position)
-   Place market making orders at reservation price ± half spread positions
-   Does not consider order book imbalance (OBI) signals, focuses on pure market making logic

### BacktestAsset Configuration Description

Configure BacktestAsset using method chaining:

```python
data_asset = (
    BacktestAsset()
    .data([data_file])              # Data file list
    .linear_asset(1.0)              # Linear asset, contract multiplier
    .risk_adverse_queue_model()     # Queue model
    .no_partial_fill_exchange()     # Exchange model
    .tick_size(0.01)               # Price precision
    .lot_size(0.001)               # Minimum trade size
    .trading_value_fee_model(-0.00003, 0.0007)  # Fee model
)
```

In [ ]:
# Simple training loop example
print("🎯 Simple training loop example...")

try:
    # Define action space boundaries
    # [half_spread, skew(tick count), grid_num, grid_interval_multiplier]
    action_low = [1.0, 1.0, 1.0, 1.0]
    action_high = [50.0, 50.0, 10.0, 50.0]

    max_steps = 3600  # Maximum 3600 strategy executions per episode
    risk_penalty_weight = 0.01
    step_interval_ns = 500_000_000  # 0.5 seconds between each strategy execution

    # Create pure PnL environment instance
    env = create_pmm_env(
        data_asset=data_asset,
        action_low=action_low,
        action_high=action_high,
        max_steps=max_steps,
        device=device.type,
        risk_penalty_weight=risk_penalty_weight,
        step_interval_ns=step_interval_ns,  # 0.5 seconds between each strategy execution
    )

    # =================== Episode and Steps Relationship Explanation ===================
    # Episode (Round): A complete training cycle from initial state to end condition
    # Steps (Step count): Number of strategy executions within an episode
    #
    # Hierarchy:
    # - Training process contains multiple episodes (3 here)
    # - Each episode contains multiple steps (max_steps at most)
    # - Each step executes strategy once (adjust parameters, place orders, wait 0.5 seconds)
    #
    # Specific numbers:
    # - num_episodes = 3: Conduct 3 independent training rounds
    # - max_steps = 3600: Strategy executes at most 3600 times per episode
    # - step_interval_ns = 0.5 seconds: Each strategy execution advances time by 0.5 seconds
    # - Simulation time per episode = max_steps × 0.5 seconds = 1800 seconds (30 minutes)
    # - Total strategy executions = num_episodes × max_steps = 10800 times
    # ==================================================================

    # Run multiple episodes (training rounds)
    num_episodes = 3  # Run 3 independent training rounds
    episode_rewards = []

    for episode in range(num_episodes):
        print(f"\n📍 Episode {episode + 1}/{num_episodes}")

        # ============ Episode Start ============
        # Each episode is an independent training round:
        # 1. Environment reset to initial state (PnL=0, position=0)
        # 2. Data reads from beginning
        # 3. All states reset to zero
        obs_td = env.reset()
        episode_reward = 0
        step_count = 0

        # ============ Steps Loop Within Episode ============
        # Within this episode, strategy will execute multiple times (steps)
        # Each execution (step) includes:
        # 1. Select action based on current observation (adjust strategy parameters)
        # 2. Execute strategy (place orders, cancel orders, etc.)
        # 3. Advance time by 0.5 seconds, process market events
        # 4. Get reward, update state
        while True:
            # Step 1: Generate action (strategy parameters)
            # In actual training, this would be replaced by SAC policy network output
            # Here we use random actions for demonstration
            random_values = torch.rand(4, device=env.device)
            action_tensor_low = torch.tensor(action_low, device=env.device)
            action_tensor_high = torch.tensor(action_high, device=env.device)

            # Generate random actions within action space bounds
            action = action_tensor_low + random_values * \
                (action_tensor_high - action_tensor_low)

            # Step 2: Execute action (strategy execution once)
            # This will:
            # - Update strategy parameters (half_spread, skew, etc.)
            # - Call pure_pmm_step to execute strategy
            # - Advance time by 0.5 seconds (step_interval_ns)
            # - Process all market events within this 0.5 seconds
            action_td = TensorDict(
                {"action": action}, batch_size=(), device=env.device
            )
            step_td = env.step(action_td)  # Strategy executes once, time advances by 0.5 seconds

            # Step 3: Get reward from this execution
            if 'reward' in step_td:
                reward = step_td['reward'].item()
            elif 'next' in step_td and 'reward' in step_td['next']:
                reward = step_td['next']['reward'].item()
            else:
                reward = 0

            episode_reward += reward  # Accumulate total reward for this episode
            step_count += 1  # Strategy execution count +1

            # Step 4: Check if episode is done
            # End conditions:
            # - Data exhausted (most common)
            # - Reached max_steps
            # - PnL loss exceeds threshold
            if 'done' in step_td:
                done = step_td['done'].item()
                next_obs = step_td.get('observation', obs_td['observation'])
            elif 'next' in step_td:
                done = step_td['next']['done'].item(
                ) if 'done' in step_td['next'] else False
                next_obs = step_td['next'].get(
                    'observation', obs_td['observation'])
            else:
                done = False
                next_obs = obs_td['observation']

            if done:
                break  # This episode ends

            # Step 5: Update observation, continue to next step
            obs_td = TensorDict(
                {"observation": next_obs}, batch_size=(), device=env.device
            )
            # Loop continues, execute next strategy (next step)

        # ============ Episode End ============
        episode_rewards.append(episode_reward)

        # Get final state and display statistics
        final_state = env._get_strategy_state()
        final_pnl = final_state['pnl']  # Direct PnL, starting from 0

        print(f"  Steps: {step_count}")  # How many times strategy executed in this episode
        print(f"  Simulation duration: {step_count * 0.5}s ({step_count * 0.5 / 60:.1f} minutes)")  # How much market time simulated
        print(f"  Total reward: {episode_reward:.4f}")
        print(f"  Final PnL: ${final_pnl:.2f}")
        print(f"  Return rate: {final_pnl / 50.0:.8%}")  # Return based on $50 order amount

    # ============ All Episodes Complete ============
    # Training summary
    print(f"\n📊 Training Summary:")
    print(f"  Ran {num_episodes} episodes")
    print(f"  Maximum {max_steps} steps per episode (strategy execution count)")
    print(f"  Total strategy executions: approximately {num_episodes * max_steps}")
    print(f"  Average reward: {sum(episode_rewards) / len(episode_rewards):.4f}")
    print(f"  Best reward: {max(episode_rewards):.4f}")
    print(f"  Worst reward: {min(episode_rewards):.4f}")

    env.close()
    print("\n✅ Training loop example completed!")

except Exception as e:
    print(f"❌ Training loop failed: {e}")
    import traceback
    traceback.print_exc()